## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 📂 2. Load Processed Data from Module 1

In [ ]:
# Load the data with categories from Module 1
train_data = pd.read_csv('/content/dataset/KDDTrain+.txt', header=None)
test_data = pd.read_csv('/content/dataset/KDDTest+.txt', header=None)

# Add column names
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

train_data.columns = column_names
test_data.columns = column_names

print(f"✅ Training Data Loaded: {train_data.shape}")
print(f"✅ Test Data Loaded: {test_data.shape}")

## 📝 3. Preprocessing Technique #1: Lowercasing & Cleaning Textual Features

**Goal:** Standardize textual data by converting to lowercase and removing unwanted characters

In [ ]:
# Identify textual columns
textual_columns = ['protocol_type', 'service', 'flag', 'label']

print("🔄 BEFORE Cleaning:")
print("="*60)
for col in textual_columns:
    print(f"{col:20s}: {train_data[col].nunique()} unique values")
    print(f"  Sample: {list(train_data[col].unique()[:5])}")
    print()

# Apply lowercasing and cleaning
def clean_text(text):
    """
    Clean text by:
    1. Converting to lowercase
    2. Removing extra spaces
    3. Removing special characters (keeping alphanumeric and underscores)
    """
    text = str(text).lower().strip()
    # Remove any leading/trailing whitespace
    text = ' '.join(text.split())
    return text

# Apply cleaning to both train and test data
for col in textual_columns:
    train_data[col] = train_data[col].apply(clean_text)
    test_data[col] = test_data[col].apply(clean_text)

print("\n✅ AFTER Cleaning:")
print("="*60)
for col in textual_columns:
    print(f"{col:20s}: {train_data[col].nunique()} unique values")
    print(f"  Sample: {list(train_data[col].unique()[:5])}")
    print()

print("\n🎯 Technique #1 Completed: Lowercasing & Cleaning")

## 🔢 4. Preprocessing Technique #2: Label Encoding for Textual Features

**Goal:** Convert textual categorical features into numerical format for Deep Learning

In [ ]:
# Create label encoders for each textual column
label_encoders = {}

print("🔄 Encoding Textual Features to Numerical:")
print("="*60)

for col in ['protocol_type', 'service', 'flag']:
    # Initialize encoder
    le = LabelEncoder()
    
    # Fit on combined train+test to ensure all categories are captured
    combined = pd.concat([train_data[col], test_data[col]])
    le.fit(combined)
    
    # Transform both datasets
    train_data[f'{col}_encoded'] = le.transform(train_data[col])
    test_data[f'{col}_encoded'] = le.transform(test_data[col])
    
    # Store encoder for later use
    label_encoders[col] = le
    
    print(f"{col:20s}:")
    print(f"  Original values: {list(train_data[col].unique()[:3])}")
    print(f"  Encoded values:  {list(train_data[f'{col}_encoded'].unique()[:3])}")
    print(f"  Encoding range:  0 to {train_data[f'{col}_encoded'].max()}")
    print()

print("\n🎯 Technique #2 Completed: Label Encoding")

In [ ]:
# Create mapping visualization for protocol_type
print("\n📊 Encoding Mappings:")
print("="*60)

for col in ['protocol_type', 'service', 'flag']:
    mapping = dict(zip(label_encoders[col].classes_, 
                      label_encoders[col].transform(label_encoders[col].classes_)))
    print(f"\n{col.upper()} Mapping:")
    for original, encoded in sorted(mapping.items(), key=lambda x: x[1])[:10]:
        print(f"  '{original}' → {encoded}")
    if len(mapping) > 10:
        print(f"  ... and {len(mapping) - 10} more")

## 🎯 5. Encode Target Labels (Attack Categories)

**Create binary and multi-class target variables**

In [ ]:
# Categorize attacks
dos_attacks = ['neptune', 'smurf', 'pod', 'teardrop', 'land', 'back', 'apache2', 'udpstorm', 'processtable', 'mailbomb']
probe_attacks = ['satan', 'ipsweep', 'nmap', 'portsweep', 'mscan', 'saint']
r2l_attacks = ['guess_passwd', 'ftp_write', 'imap', 'phf', 'multihop', 'warezmaster', 'warezclient', 'spy', 'xlock', 'xsnoop', 'snmpguess', 'snmpgetattack', 'httptunnel', 'sendmail', 'named']
u2r_attacks = ['buffer_overflow', 'loadmodule', 'rootkit', 'perl', 'sqlattack', 'xterm', 'ps']

def categorize_attack(label):
    if label == 'normal':
        return 'Normal'
    elif label in dos_attacks:
        return 'DoS'
    elif label in probe_attacks:
        return 'Probe'
    elif label in r2l_attacks:
        return 'R2L'
    elif label in u2r_attacks:
        return 'U2R'
    else:
        return 'Other'

# Apply categorization
train_data['attack_category'] = train_data['label'].apply(categorize_attack)
test_data['attack_category'] = test_data['label'].apply(categorize_attack)

# Binary target: Normal (0) vs Attack (1)
train_data['is_attack'] = (train_data['label'] != 'normal').astype(int)
test_data['is_attack'] = (test_data['label'] != 'normal').astype(int)

# Multi-class target: Encode attack categories
category_encoder = LabelEncoder()
combined_categories = pd.concat([train_data['attack_category'], test_data['attack_category']])
category_encoder.fit(combined_categories)

train_data['attack_category_encoded'] = category_encoder.transform(train_data['attack_category'])
test_data['attack_category_encoded'] = category_encoder.transform(test_data['attack_category'])

print("🎯 Target Variable Encoding:")
print("="*60)
print(f"\n1. Binary Classification (Normal vs Attack):")
print(train_data['is_attack'].value_counts())

print(f"\n2. Multi-Class Classification (Attack Categories):")
print(train_data['attack_category'].value_counts())

print(f"\n3. Category Encoding Mapping:")
for cat, encoded in zip(category_encoder.classes_, 
                        category_encoder.transform(category_encoder.classes_)):
    print(f"  '{cat}' → {encoded}")

# Store encoder
label_encoders['attack_category'] = category_encoder

## 📊 6. Numerical Feature Preprocessing: Scaling & Normalization

**Goal:** Normalize numerical features to improve Deep Learning training

In [ ]:
# Identify numerical columns (exclude text and encoded text columns)
exclude_cols = ['protocol_type', 'service', 'flag', 'label', 'difficulty', 
                'attack_category', 'is_attack', 'attack_category_encoded',
                'protocol_type_encoded', 'service_encoded', 'flag_encoded']

numerical_cols = [col for col in train_data.columns if col not in exclude_cols]

print("🔢 Numerical Features to Normalize:")
print("="*60)
print(f"Total numerical features: {len(numerical_cols)}")
print(f"\nFeatures: {numerical_cols[:10]}...")

# Show statistics before normalization
print("\n📊 BEFORE Normalization (sample features):")
print(train_data[numerical_cols[:5]].describe())

In [ ]:
# Apply StandardScaler (zero mean, unit variance)
scaler = StandardScaler()

# Fit on training data only
scaler.fit(train_data[numerical_cols])

# Transform both train and test
train_data_scaled = train_data.copy()
test_data_scaled = test_data.copy()

train_data_scaled[numerical_cols] = scaler.transform(train_data[numerical_cols])
test_data_scaled[numerical_cols] = scaler.transform(test_data[numerical_cols])

print("✅ StandardScaler Applied!")
print("="*60)
print("\n📊 AFTER Normalization (sample features):")
print(train_data_scaled[numerical_cols[:5]].describe())

print("\n🎯 Normalization Complete!")
print(f"  Mean ≈ 0, Std ≈ 1 for all features")

## 📈 7. Visualize Preprocessing Effects

In [ ]:
# Compare before and after normalization
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: Before vs After Normalization', fontsize=16, fontweight='bold')

sample_features = numerical_cols[:3]

for idx, feature in enumerate(sample_features):
    # Before normalization
    axes[0, idx].hist(train_data[feature], bins=50, color='coral', alpha=0.7, edgecolor='black')
    axes[0, idx].set_title(f'{feature}\n(Before)', fontsize=10)
    axes[0, idx].set_ylabel('Frequency')
    
    # After normalization
    axes[1, idx].hist(train_data_scaled[feature], bins=50, color='skyblue', alpha=0.7, edgecolor='black')
    axes[1, idx].set_title(f'{feature}\n(After Normalization)', fontsize=10)
    axes[1, idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 🔍 8. Create Final Feature Matrix for Deep Learning

In [ ]:
# Select features for modeling
# Use encoded textual features + normalized numerical features
feature_columns = numerical_cols + ['protocol_type_encoded', 'service_encoded', 'flag_encoded']

# Create feature matrices
X_train = train_data_scaled[feature_columns].values
X_test = test_data_scaled[feature_columns].values

# Create target vectors
# Binary classification
y_train_binary = train_data_scaled['is_attack'].values
y_test_binary = test_data_scaled['is_attack'].values

# Multi-class classification
y_train_multi = train_data_scaled['attack_category_encoded'].values
y_test_multi = test_data_scaled['attack_category_encoded'].values

print("🎯 Final Preprocessed Data Shapes:")
print("="*70)
print(f"\n📊 Feature Matrix:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")

print(f"\n🎯 Binary Classification Target:")
print(f"  y_train shape: {y_train_binary.shape}")
print(f"  y_test shape:  {y_test_binary.shape}")
print(f"  Classes: {np.unique(y_train_binary)}")

print(f"\n🎯 Multi-Class Classification Target:")
print(f"  y_train shape: {y_train_multi.shape}")
print(f"  y_test shape:  {y_test_multi.shape}")
print(f"  Classes: {np.unique(y_train_multi)} (0=DoS, 1=Normal, 2=Other, 3=Probe, 4=R2L, 5=U2R)")

## 📋 9. Module 2 Summary Report

In [ ]:
print("\n" + "="*70)
print("📋 MODULE 2: PREPROCESSING - SUMMARY REPORT")
print("="*70)

print("\n✅ PREPROCESSING TECHNIQUES APPLIED:")
print("\n1️⃣ Textual Preprocessing:")
print("   ✓ Technique #1: Lowercasing & Cleaning")
print("     - Converted all text to lowercase")
print("     - Removed extra whitespace")
print("     - Standardized text format")

print("\n   ✓ Technique #2: Label Encoding")
print(f"     - Encoded protocol_type: {train_data['protocol_type'].nunique()} → 0-{train_data['protocol_type_encoded'].max()}")
print(f"     - Encoded service: {train_data['service'].nunique()} → 0-{train_data['service_encoded'].max()}")
print(f"     - Encoded flag: {train_data['flag'].nunique()} → 0-{train_data['flag_encoded'].max()}")

print("\n2️⃣ Numerical Preprocessing:")
print("   ✓ StandardScaler Normalization")
print(f"     - Normalized {len(numerical_cols)} numerical features")
print("     - Mean ≈ 0, Standard Deviation ≈ 1")
print("     - Improved numerical stability for Deep Learning")

print("\n3️⃣ Target Encoding:")
print("   ✓ Binary Classification: Normal (0) vs Attack (1)")
print(f"     - Training: {y_train_binary.shape[0]:,} samples")
print(f"     - Testing: {y_test_binary.shape[0]:,} samples")

print("\n   ✓ Multi-Class Classification: 6 Attack Categories")
print(f"     - DoS, Normal, Probe, R2L, U2R, Other")
print(f"     - Training: {y_train_multi.shape[0]:,} samples")
print(f"     - Testing: {y_test_multi.shape[0]:,} samples")

print("\n📊 FINAL DATA READY FOR MODULE 3:")
print(f"   📈 Feature Matrix: {X_train.shape[1]} features")
print(f"   🎯 Binary Target: 2 classes (Normal/Attack)")
print(f"   🎯 Multi-Class Target: 6 classes (Attack Categories)")

print("\n" + "="*70)
print("✅ MODULE 2 COMPLETED SUCCESSFULLY!")
print("🚀 Ready for Module 3: Deep Learning Model Building")
print("="*70)

## 💾 10. Save Preprocessed Data for Module 3

In [ ]:
# Save preprocessed data as numpy arrays
np.save('/content/X_train.npy', X_train)
np.save('/content/X_test.npy', X_test)
np.save('/content/y_train_binary.npy', y_train_binary)
np.save('/content/y_test_binary.npy', y_test_binary)
np.save('/content/y_train_multi.npy', y_train_multi)
np.save('/content/y_test_multi.npy', y_test_multi)

# Save feature names
with open('/content/feature_names.txt', 'w') as f:
    for feature in feature_columns:
        f.write(f"{feature}\n")

print("\n💾 Preprocessed Data Saved Successfully!")
print("="*60)
print("✅ X_train.npy")
print("✅ X_test.npy")
print("✅ y_train_binary.npy")
print("✅ y_test_binary.npy")
print("✅ y_train_multi.npy")
print("✅ y_test_multi.npy")
print("✅ feature_names.txt")
print("\n🚀 Ready for Module 3: Model Building!")